<div dir="rtl">
<h1>دو ویژگی، یک Neuron و یک اصلاح واقعی</h1>
<p>درس 16 از 76 · یک Neuron را با دست آموزش دهیم · <code dir="ltr">12b-neuron</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-02/chapter-06/12b-neuron.html">📖 بازگشت به همین درس</a></p>
<p>Forward و Gradient دستی یک Neuron با ReLU را بسازید.</p><p>پیش‌نیاز: جمع وزن‌دار،12-chain و12-sgd؛ فقط Python.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>برای x=[1,2]، w=[0.5,-0.25] و Bias=1، پیش‌بینی چیست؟ اگر فقط Bias را -1 کنیم، آیا Loss غیرصفر حتماً Gradient غیرصفر می‌دهد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
x, w, bias, target = [1., 2.], [0.5, -0.25], 1., 2.
print("Input:", x, "Weights:", w, "Bias:", bias, "Target:", target)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع neuron(inputs,weights,Bias) tupleِ (z, Prediction) برگرداند: z جمع وزن‌دار همراه Bias و Prediction برابر ReLU آن است. اندازه‌های نابرابر را رد کنید.</p>
</div>

In [ ]:
def neuron(inputs, weights, bias):
    # TODO: return preactivation and prediction
    return None

In [ ]:
def test_exercise():
    result = neuron(x, w, bias)
    if result is None:
        return False
    assert result == (1., 1.)
    assert neuron(x, w, -1.) == (-1., 0.)
    assert neuron([0, 0], w, 3) == (3, 3)
    try:
        neuron([1], [2, 3], 0)
    except ValueError:
        pass
    else:
        raise AssertionError("Feature and weight counts differ")
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Target را از ۲ به ۳ تغییر دهید؛ x,w,b ثابت‌اند. پیش‌بینی را یک بار بسازید و Loss هر دو هدف را مقایسه کنید.</p>
</div>

In [ ]:
prediction = max(0., sum(a*b for a, b in zip(x, w))+bias)
for goal in [2., 3.]:
    print("Target:", goal, "Prediction:", prediction, "Loss:", (prediction-goal)**2)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>نسخهٔ خراب حتی در z منفی شیب ReLU را یک می‌گیرد. تابع neuron_gradients باید (grad_weights, grad_bias) را برای Loss مربع خطا برگرداند. در z=0 به‌جای فرض مشتق، ValueError بدهید.</p>
</div>

In [ ]:
negative_z = sum(a*b for a, b in zip(x, w))-1
wrong_grad = 2*(max(0, negative_z)-target)
print("z:", negative_z, "Broken bias gradient:", wrong_grad)
assert negative_z == -1 and wrong_grad == -4

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def neuron_gradients(inputs, weights, bias, target):
    # TODO: include the local ReLU derivative; reject z=0
    return None

In [ ]:
def test_repair():
    result = neuron_gradients(x, w, 1., 2.)
    if result is None:
        return False
    assert result == ([-2., -4.], -2.)
    assert neuron_gradients(x, w, -1., 2.) == ([0., 0.], 0.)
    new_w = [weight-0.1*gradient for weight, gradient in zip(w, result[0])]
    new_b = bias-0.1*result[1]
    assert math.isclose(max(0, sum(a*b for a, b in zip(x, new_w))+new_b), 2.2)
    try:
        neuron_gradients([0], [1], 0, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("The two-sided derivative at zero is not defined")
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>Layer‌های mini_gpt/transformer.py نیز جمع وزن‌دار و Activation دارند، اما از GELU استفاده می‌کنند. این Neuron نسخهٔ کوچک برای فهم مسیر است، نه کپی معماری GPT.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>کدام مشاهده نشان داد Gradient صفر با «پاسخ درست» یکی نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-02/chapter-06/12b-neuron.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/12b-neuron.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>